# Data Engineering Lifecycle

The **Data Engineering Lifecycle** (from *Fundamentals of Data Engineering* – Reis & Housley) describes the stages data moves through from raw form to actionable insights. It is the **core framework** that data engineers use to think about their work.

<img src="https://www.databricks.com/wp-content/uploads/2021/09/Databricks-Blog_Data-Engineering-Lifecycle.png" alt="Data Engineering Lifecycle" width="720"/>

> *Source: Databricks / Fundamentals of Data Engineering – Reis & Housley*

**Undercurrents** (cross-cutting concerns that apply to every stage):
- Security
- Data Management
- DataOps
- Data Architecture
- Orchestration
- Software Engineering

## Stage 1 – Source Systems

The **origin** of data before it enters the data engineering lifecycle.

| Source Type | Description | Examples |
|-------------|-------------|----------|
| **Databases** | Operational/transactional systems | MySQL, PostgreSQL, MongoDB |
| **APIs** | Programmatic data access | REST APIs, GraphQL |
| **Event streams** | Continuous event data | Kafka, Kinesis, IoT sensors |
| **Files** | Periodic file drops | CSV, JSON, Parquet, logs |
| **SaaS platforms** | Cloud applications | Salesforce, Google Analytics |

### Key questions to ask about source systems:
- What is the **schema** of the data? Is it fixed or flexible?
- What is the **volume** and **velocity** of data?
- How is the data updated? Are deletions soft or hard?
- What happens if the source is unavailable?

## Stage 2 – Ingestion

The process of **moving data from source systems into a storage layer**. The first step in the lifecycle where data engineers have direct control.

### Batch vs. Streaming Ingestion

| Aspect | Batch Ingestion | Streaming Ingestion |
|--------|----------------|---------------------|
| **Trigger** | Scheduled interval (hourly, daily) | Continuous / event-driven |
| **Latency** | High (minutes to hours) | Low (milliseconds to seconds) |
| **Complexity** | Simpler to implement | More complex |
| **Use case** | End-of-day reports, data warehouse loads | Fraud detection, live dashboards |
| **Databricks tool** | Spark batch jobs, Auto Loader | Structured Streaming, Kafka connector |

### Common Patterns
- **CDC (Change Data Capture)**: Tracks only the changes (inserts/updates/deletes) in source databases. Tools: Debezium, Delta Live Tables.
- **Full load**: Re-ingests the complete source dataset every run.
- **Incremental load**: Ingests only records that changed since the last run (watermark-based).

### Push vs. Pull
- **Push**: The source system sends data to the ingestion layer (e.g., webhooks, Kafka producers).
- **Pull**: The ingestion layer fetches data from the source on a schedule (e.g., JDBC polling, API calls).

## Stage 3 – Storage

Storage underpins **every stage** of the lifecycle. Choosing the right storage depends on access patterns, cost, latency, and query type.

### Storage Abstractions

| Abstraction | Description | Databricks Context |
|-------------|-------------|-------------------|
| **Data Lake** | Raw storage of all data in open formats (no schema enforcement) | ADLS Gen2, S3, GCS |
| **Data Warehouse** | Structured, schema-on-write, optimized for SQL analytics | Redshift, Snowflake, Synapse |
| **Data Lakehouse** | Combines lake flexibility with warehouse reliability | **Delta Lake on Databricks** |
| **Object Storage** | Scalable, cheap, unmanaged storage | Azure Blob, AWS S3 |
| **OLTP Database** | Optimized for transactional operations | PostgreSQL, MySQL |

### Storage Considerations
- **Serialization formats**: Parquet (columnar, compressed), Avro (row-based, schema evolution), ORC, Delta (Parquet + transaction log).
- **Partitioning**: Organizing files by a column value (e.g., `date`) to reduce scan costs.
- **ACID transactions**: Guaranteed data integrity — Atomicity, Consistency, Isolation, Durability. Enabled by **Delta Lake**.

## Stage 4 – Transformation

The process of **converting raw, ingested data into clean, enriched, and structured data** ready for downstream consumption.

### Medallion Architecture (Delta Lake / Databricks standard)

```
Bronze (Raw)  →  Silver (Cleaned/Validated)  →  Gold (Aggregated/Business-ready)
```

| Layer | Also Called | Description | Example |
|-------|------------|-------------|----------|
| **Bronze** | Raw / Landing | Exact copy of source data, no changes | Raw JSON from Kafka |
| **Silver** | Cleaned / Enriched | Deduplicated, validated, joined | Customer + orders joined |
| **Gold** | Curated / Aggregated | Business KPIs, ready for BI/ML | Daily revenue by region |

### Transformation Patterns
- **ELT (Extract-Load-Transform)**: Modern approach – load raw first, transform inside the platform. Preferred in Databricks/lakehouse environments.
- **ETL (Extract-Transform-Load)**: Traditional approach – transform before loading into the warehouse.
- **Data modeling**: Star schema, snowflake schema, OBT (One Big Table).

### Transformation Tools (Databricks Focused)
| Tool | Use |
|------|-----|
| **PySpark** | Distributed data transformation at scale |
| **Spark SQL** | SQL-based transformations on DataFrames |
| **Delta Live Tables (DLT)** | Declarative ETL pipelines with data quality checks |
| **dbt (with Databricks)** | SQL-based transformation and data modeling |

## Stage 5 – Serving

The stage where transformed data is made **available to end users and systems** — analysts, data scientists, applications.

### Serving Use Cases

| Use Case | Description | Databricks Tool |
|----------|-------------|------------------|
| **Analytics / BI** | Dashboards, reports, ad-hoc queries | SQL Warehouse, Power BI, Tableau |
| **Machine Learning** | Feature engineering, model training | MLflow, Feature Store |
| **Reverse ETL** | Pushing insights back to operational tools | Census, Hightouch |
| **Operational Analytics** | Real-time decisions embedded in apps | Delta + Structured Streaming |

### Data Products
> *"Data products are the outputs of the data engineering lifecycle that add value to the business."*
- Reports and dashboards
- Trained ML models
- Enriched datasets published to other teams
- APIs powered by curated data

## Undercurrents of the Data Engineering Lifecycle

These cross-cutting concerns apply to **all stages** of the lifecycle.

| Undercurrent | Description |
|--------------|-------------|
| **Security** | Least-privilege access, encryption at rest and in transit, masking PII. Databricks: Unity Catalog, RBAC, column masking. |
| **Data Management** | Governance, cataloging, lineage, data quality. Databricks: Unity Catalog. |
| **DataOps** | Applying DevOps principles to data pipelines — CI/CD, monitoring, alerting, testing. |
| **Data Architecture** | Designing systems that are scalable, reliable, and cost-effective (see notebook 06). |
| **Orchestration** | Scheduling and managing pipeline dependencies. Tools: Apache Airflow, Databricks Workflows, dbt Cloud. |
| **Software Engineering** | Writing maintainable, testable, modular code for pipelines. |

### DataOps Practices
- **Version control** for pipelines (Git + Databricks Repos)
- **Automated testing**: unit tests on transformations, data quality checks (Great Expectations, DLT expectations)
- **Monitoring and alerting**: track job failures, data freshness, schema drift
- **CI/CD pipelines**: auto-deploy on merge to main